# EasternDryRocks intake inventory

This notebook is the hands-on companion to `reef-sfm validate-intake`.
It runs the same code path the CLI uses, but exposes the intermediate
data frames so we can sanity-check the image set before we commit to the
(long, expensive) Metashape run in Chat 5.

**Expected runtime:** ~3-5 minutes for ~2000 images on the EC2 g6.4xlarge.
**Expected outcome:** all per-image checks pass; dataset-level checks pass
or warn (`gps_consistency` warns harmlessly if the EXIF rounding pushed the
per-site coordinate cluster over the 5m tolerance).

If anything fails, the QC report at the end says exactly which files and which rules.

In [ ]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import pandas as pd

from reef_sfm_provenance.acquisition import load_provenance
from reef_sfm_provenance.contact_sheet import generate_contact_sheets
from reef_sfm_provenance.ids_csv import IDS_CSV_DEFAULT, load_ids_csv
from reef_sfm_provenance.intake_report import (
    build_report, write_report_json, write_report_markdown,
)
from reef_sfm_provenance.inventory import build_inventory, iter_image_paths
from reef_sfm_provenance.validation import validate_dataset, validate_image

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

## Paths

On the EC2 instance the data volume is mounted at `/data` (per Chat 2's
`/etc/fstab` config).  Acquired images live under
`/data/raw/P1WHKTRD/<site>/`.  Adjust the constant below if running locally.

In [ ]:
DATA_ROOT = Path('/data/raw/P1WHKTRD')
SITE = 'EasternDryRocks'
SITE_DIR = DATA_ROOT / SITE
FIGURES_DIR = Path(__file__).resolve().parent.parent / 'figures' if '__file__' in dir() else Path('../figures')
# IDS viewer CSV — path relative to the project root (tracked in git)
IDS_CSV_PATH = Path('data/reference/ids_export/exif_data.csv')

print(f'Site dir:    {SITE_DIR}')
print(f'Exists:      {SITE_DIR.exists()}')
print(f'IDS CSV:     {IDS_CSV_PATH} (exists: {IDS_CSV_PATH.exists()})')
print(f'Image count: {sum(1 for _ in iter_image_paths(SITE_DIR)) if SITE_DIR.exists() else 0}')

## Load acquisition provenance

If the `acquire` step has run, every image already has a SHA-256 computed
during streaming download.  We reuse those hashes in the inventory rather
than re-reading the files.

In [ ]:
prov_path = SITE_DIR / '_provenance.json'
if prov_path.exists():
    prov = load_provenance(prov_path)
    hashes_by_name = {f['name']: f['sha256'] for f in prov['files']}
    print(f"Loaded {len(hashes_by_name)} hashes from acquisition provenance")
    print(f"Acquisition timestamp: {prov['downloaded_at_utc']}")
else:
    hashes_by_name = {}
    print('No acquisition provenance yet — run `reef-sfm acquire` first.')

In [ ]:
ids_path = IDS_CSV_PATH if IDS_CSV_PATH.exists() else None
if ids_path:
    ids_records = load_ids_csv(ids_path)
    print(f"IDS CSV loaded: {len(ids_records):,} rows from {ids_path}")
else:
    ids_records = {}
    print("Warning: IDS CSV not found — CSV-primary checks will be unverified."
          f"  Expected: {IDS_CSV_PATH}")

## Build the inventory

EXIF via Pillow, XMP/IPTC via `exiftool` if it's on PATH.

In [ ]:
inv = build_inventory(SITE_DIR, hashes_by_name=hashes_by_name, ids_records=ids_records)
print(f'Inventoried {len(inv)} images')

In [ ]:
# Show the head as a DataFrame for quick visual scanning
df = pd.DataFrame([rec.to_dict() for rec in inv])
df[['name', 'width', 'height', 'exif_make', 'exif_model',
    'exif_datetime_original', 'gps_lat', 'gps_lon', 'size_bytes']].head(10)

In [ ]:
# Distribution summary — confirms camera consistency and GPS clustering at a glance
print('Cameras observed:')
print(df.groupby(['exif_make', 'exif_model']).size())
print()
print('GPS spread (decimal degrees):')
print(df[['gps_lat', 'gps_lon']].describe())
print()
print('File size distribution (MB):')
print((df['size_bytes'] / (1 << 20)).describe())

## Run the validation rules

In [ ]:
per_image_findings = []
for rec in inv:
    per_image_findings.extend(validate_image(rec))

dataset_findings = validate_dataset(inv)

# Quick rollup
f_df = pd.DataFrame([f.to_dict() for f in per_image_findings])
if not f_df.empty:
    print(pd.crosstab(f_df['code'], f_df['severity']))

In [ ]:
print('Dataset-level findings:')
for f in dataset_findings:
    print(f'  [{f.severity:>10}] {f.code}: {f.message}')

## Write the report

In [ ]:
report = build_report(
    site=SITE,
    doi='10.5066/P1WHKTRD',
    site_dir=SITE_DIR,
    inventory=inv,
    dataset_findings=dataset_findings,
    per_image_findings=per_image_findings,
    ids_csv_path=str(ids_path) if ids_path else None,
)

write_report_json(report, SITE_DIR / 'qc_report.json')
write_report_markdown(report, SITE_DIR / 'qc_report.md')

print(f"Overall severity: {report['overall_severity']}")
print(f"Wrote: {SITE_DIR / 'qc_report.md'}")

## Contact sheets

JPEG thumbnail grids for visual review.  ~50 sheets at 6x6 thumbnails
for a 2000-image set.

In [ ]:
contact_dir = FIGURES_DIR / 'contact_sheets' / SITE
sheets = generate_contact_sheets(
    list(iter_image_paths(SITE_DIR)),
    contact_dir,
    cols=6, rows=6, tile_w=220, tile_h=165,
)
print(f'Wrote {len(sheets)} contact sheets to {contact_dir}')